In [1]:
from flask import Flask, jsonify, request
from flask_cors import CORS
import pymysql  # or import mysql.connector
import joblib
import pandas as pd
import numpy as np
# calories, protien, sugar, fat, fiber, carbohydrates

In [2]:
model = joblib.load("SleepAnalysis.pkl")
# /home/kali/College/Mini/Job/SleepAnalysis.pkl
# SleepAnalysis2.pkl
db = pymysql.connect(
host = "localhost",
user = "root", #root #aditya
password = "root",
database = "mini"
)
cursor = db.cursor()

In [ ]:
app = Flask(__name__)
cors = CORS(app, origins = '*')
@app.route("/submit", methods = ['GET', 'POST'] )
def submit():
    data = request.get_json()
    age = int(data['age'])
    bed_time = data['bedTime']
    wake_time = data['wakeTime']
    awakenings = float(data['awakenings'])
    caffeine = float(data['caffeine'])
    alcohol = float(data['alcohol'])
    smoking = "Yes" if data['smoking'].lower() == "yes" else "No"  # Store as Yes/No
    exercise = float(data['exercise'])
    REM = int(data['REM'])
    deep_sleep = int(data['deep_sleep'])


    smoking_numeric = 1 if smoking == "Yes" else 0
    sleep_duration = (float(wake_time.split(":")[0]) - float(bed_time.split(":")[0]) + 24) % 24

    userDataDF = np.array([[  age,
 sleep_duration,
    REM, 
  deep_sleep, 
awakenings,
 caffeine,
 alcohol,
 smoking_numeric,  
exercise]])
    # userDataDF = pd.DataFrame({
    #     'Age': [age],
    #     'Sleep_duration': [sleep_duration],
    #     'REM_sleep_percentage': [REM], 
    #     'Deep_sleep_percentage': [deep_sleep], 
    #     'Awakenings': [awakenings],
    #     'Caffeine_consumption': [caffeine],
    #     'Alcohol_consumption': [alcohol],
    #     'Smoking_status': [smoking_numeric],  
    #     'Exercise_frequency': [exercise]
    # })
    prediction = model.predict(userDataDF)
    sleep_efficiency = prediction[0]
    insert_query = """
        INSERT INTO sleep_data (age, bed_time, wake_time, awakenings, caffeine, alcohol, smoking, exercise, sleep_efficiency, REM_percentage, deep_sleep_percentage) 
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    values = (age, bed_time, wake_time, awakenings, caffeine, alcohol, smoking, exercise, sleep_efficiency, REM, deep_sleep)
    cursor.execute(insert_query, values)
    db.commit()
    print(data)
    return jsonify({'sleep_efficiency': prediction[0], "duration": sleep_duration, "data" : data, "values": values })

@app.route("/diet", methods = ['GET', 'POST'])
def diet():
    data = request.get_json()
    food_query = f"{data['food']}%"
    cursor.execute("select name from dietdb where name like %s limit 10", (food_query,))
    results = cursor.fetchall()
    results = [i[0] for i in results]
    return jsonify({'results' : results})

@app.route("/diet/output", methods = ['GET', 'POST'])
def output():
    data = request.get_json()
    serving = int(data['serving']) / 100
    cursor.execute("select calories, protein, carbohydrate, cholesterol, total_fat, sugars from dietdb where name = %s", (data['food'],))
    result = cursor.fetchone()
    output = {
        "calories": int(result[0] * serving),
        "protein": int(result[1] * serving),
        "carbohydrate": int(result[2] * serving),
        "cholesterol": int(result[3] * serving),
        "total_fat": int(result[4] * serving),
        "sugars": int(result[5] * serving)
    }
    print(type(output))
    print(output)
    return jsonify(output)

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [24/Mar/2025 01:04:33] "OPTIONS /diet/output HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:04:33] "POST /diet/output HTTP/1.1" 200 -


<class 'dict'>
{'calories': 80, 'protein': 7, 'carbohydrate': 15, 'cholesterol': 0, 'total_fat': 0, 'sugars': 7}


127.0.0.1 - - [24/Mar/2025 01:10:36] "OPTIONS /diet HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:10:37] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:10:37] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:10:43] "OPTIONS /diet/output HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:10:43] "POST /diet/output HTTP/1.1" 200 -


<class 'dict'>
{'calories': 24, 'protein': 2, 'carbohydrate': 4, 'cholesterol': 0, 'total_fat': 0, 'sugars': 0}


...

<class 'dict'>
{'calories': 1290, 'protein': 29, 'carbohydrate': 125, 'cholesterol': 0, 'total_fat': 82, 'sugars': 77}


127.0.0.1 - - [24/Mar/2025 01:20:20] "OPTIONS /diet/output HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:20:20] "POST /diet/output HTTP/1.1" 200 -


<class 'dict'>
{'calories': 1290, 'protein': 29, 'carbohydrate': 125, 'cholesterol': 0, 'total_fat': 82, 'sugars': 77}


127.0.0.1 - - [24/Mar/2025 01:20:28] "OPTIONS /diet HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:20:29] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:20:29] "POST /diet HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:20:34] "OPTIONS /diet/output HTTP/1.1" 200 -
127.0.0.1 - - [24/Mar/2025 01:20:34] "POST /diet/output HTTP/1.1" 200 -


<class 'dict'>
{'calories': 62, 'protein': 4, 'carbohydrate': 12, 'cholesterol': 0, 'total_fat': 0, 'sugars': 4}
